In [265]:
%useLatestDescriptors
%use dataframe
fun compareTwoColumns (col1: DataColumn<*>, col2: DataColumn<*>, name: String): BaseColumn<String> {
    val col = col1.mapIndexed { index, value->
        val refValue = value as Int
        val compValue = col2[index] as Int
        if (refValue <= compValue) {
            (((compValue.toDouble() / refValue.toDouble()) - 1) * 100)
        }else {
            (((refValue.toDouble() / compValue.toDouble()) - 1) * -100)
        }.toInt().toString() + "\\%"
    }
    return col.rename(name)
}

In [266]:
// static stuff
enum class Context {CLUSTER, BUDGET, CLUSTERKM, CLUSTERALL}
enum class Mode {FLAT, RANDOM}
val orderedInstances = listOf("eil101", "gil262", "pr299", "lin318", "rd400", "d493", "u574", "u724", "pcb1173", "fl1400", "pr2392").map { if (it == "instance") it else it + "-gen3-50" }


val mode = Mode.FLAT
val context = Context.CLUSTERALL


val baseline = "fbckmd"
val colsWithoutPercentages = when (context) {
    Context.CLUSTER -> baseline
    Context.BUDGET -> baseline
    Context.CLUSTERKM -> "kmn"
    Context.CLUSTERALL-> baseline
}

val colsToIgnoreWhenCalcuateMax = when (context) {
    Context.CLUSTER -> emptyList()
    Context.BUDGET -> emptyList()
    Context.CLUSTERKM -> emptyList()
    Context.CLUSTERALL -> listOf("kmn", "kmd")
}

val relativePath = "/op-solver-strict/results/${context.name.lowercase()}${mode.name.lowercase()}/comparison.csv"
val header = when (context) {
    Context.CLUSTER -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "best")
    Context.CLUSTERKM -> listOf("instance", "kmn", "kmd", "nckmn", "nckmd", "best")
    Context.CLUSTERALL -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "kmn", "kmd", "best")
    else -> null
}

val label = "${context.toString().lowercase()}_${mode.toString().lowercase()}"

val caption = when (context) {
    Context.CLUSTER -> "Comparison of the clustering results for the instances in the ${mode.name.lowercase()} mode."
    Context.BUDGET -> "Comparison of the budget results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERKM -> "Comparison of the k-means impact results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERALL -> "Clustering Algorithm Comparison for ${mode.name.lowercase()} Instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$."
}

val title = when (context) {
    Context.CLUSTER -> "clustering"
    Context.BUDGET -> "budget"
    Context.CLUSTERKM -> "k-means impact"
    Context.CLUSTERALL -> "clustering"
} + " ${mode.name.lowercase()}"

In [267]:

val mainPath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + relativePath
var df = DataFrame.readCSV(mainPath)
df = df.sortWith (compareBy { row -> orderedInstances.indexOf(row["instance"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }

df


instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd,best
eil101-gen3-50,57,57,55,57,58,59,59,59,kmd
gil262-gen3-50,135,137,129,129,139,140,131,139,fbckmd
pr299-gen3-50,149,151,147,146,148,151,154,152,kmn
lin318-gen3-50,178,183,176,172,185,184,191,186,kmn
rd400-gen3-50,204,204,193,188,207,207,208,208,kmd
d493-gen3-50,295,286,264,263,305,299,298,292,fbckmn
u574-gen3-50,303,302,295,296,309,309,313,312,kmn
u724-gen3-50,378,377,365,364,379,382,390,393,kmd
pcb1173-gen3-50,555,561,559,558,572,579,581,583,kmd
fl1400-gen3-50,966,956,791,841,979,983,1018,1020,kmd


In [268]:
val referencePath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + "/op-solver-strict/results/baseline_${mode.name.lowercase()}.csv"
val baselineDf = DataFrame.readCSV(referencePath)
    .sortWith (compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })

//baselineDf.sortBy { it["revenueAvg"] }.name.values().map { it.split("-").first() }.map { "\"$it\"" }

baselineDf

budget,budgetSpentAvg,name,revenueAvg,revenueMax,revenueMin,size,successfulAmount,timeAvg,timeMax,timeMin
315,303.139000,eil101-gen3-50,55,60,52,101,10,0.620000,1.500000,0.290000
1189,1158.178000,gil262-gen3-50,129,134,123,262,10,9.760000,36.790000,2.800000
24096,23490.985000,pr299-gen3-50,147,154,135,299,10,8.970000,36.290000,2.230000
21015,20276.351000,lin318-gen3-50,176,186,161,318,10,28.100000,125.760000,3.210000
7641,7447.897000,rd400-gen3-50,193,201,187,400,10,14.990000,66.140000,6.490000
17501,17041.166000,d493-gen3-50,264,284,228,493,10,24.690000,65.400000,6.630000
18453,18016.307000,u574-gen3-50,295,307,284,574,10,19.880000,33.420000,9.540000
20955,20427.660000,u724-gen3-50,365,381,351,724,10,44.800000,123.210000,8.520000
28446,27723.123000,pcb1173-gen3-50,559,572,528,1173,10,15.720000,21.050000,11.630000
10064,9800.178333,fl1400-gen3-50,791,919,715,1400,6,73.700000,102.720000,66.050000


In [269]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

In [270]:
df = df.remove { best }
df.update { instance }.with { it.split("-")[0] }
val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.7cm}|" }
val header = df.columnNames().joinToString(separator = " & ")

df

instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd
eil101-gen3-50,57,57,55,57,58,59,59,59
gil262-gen3-50,135,137,129,129,139,140,131,139
pr299-gen3-50,149,151,147,146,148,151,154,152
lin318-gen3-50,178,183,176,172,185,184,191,186
rd400-gen3-50,204,204,193,188,207,207,208,208
d493-gen3-50,295,286,264,263,305,299,298,292
u574-gen3-50,303,302,295,296,309,309,313,312
u724-gen3-50,378,377,365,364,379,382,390,393
pcb1173-gen3-50,555,561,559,558,572,579,581,583
fl1400-gen3-50,966,956,791,841,979,983,1018,1020


In [271]:
val bestValues = df.convert { all()}.perRowCol { row, col ->
    if (col[row] is String || colsToIgnoreWhenCalcuateMax.contains(col.name())) {
        0
    } else {
        col[row] as Int
    }
}.map{ row ->
    row.rowMaxOf<Int>()
}


val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, baselineDf[index].revenueAvg) }

val rowMinValues = df.map { row ->
    row.rowMinOf<Int>()
}

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Int>()
    (maxEntry - minEntry).toDouble() / maxEntry.toDouble()
}
val maxRevenueDif = revenueDif.max()
val gradient = 0.1 //max(maxRevenueDif,0.0)

fun getSaturation (gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100-((maxValue - value) / (maxValue * gradient) * 100)),0.0),100.0).toInt().toString()
}

bestValues


[59, 140, 151, 185, 207, 305, 309, 382, 579, 983, 1204]

In [272]:
fun calculatePercentage(refValue: Int, compValue: Int): Int{ return (((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble()) * 100).toInt()}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = df.get(colsWithoutPercentages)[row] as Int
    val percentage = calculatePercentage(refValue, compValue)
    return if (percentage >= 0) { "{\\tiny+"+percentage.toString() + "\\%}"}
    else { "{\\tiny"+percentage.toString() + "\\%}"}
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000
    } else {
        val refValue = df.get(colsWithoutPercentages)[row] as Int
        calculatePercentage(refValue, col[row] as Int)
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double && it > 0.0) {
        "+${it.roundToInt().toString()}\\%"
    } else if (it is Double){
        "${it.roundToInt().toString()}\\%"
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -2\%, -2\%, -6\%, -6\%, 0\%, -, +0\%, +0\%]

In [273]:
val stringdf = df.convert { all() }.perRowCol { row, col  ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = col[row] as Int
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())){
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        }else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = col[row] as Int
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        }else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd
eil101,\cellcolor{cyan!66} 57{\tiny-3\%},\cellcolor{cyan!66} 57{\tiny-3\%},\cellcolor{cyan!32} 55{\tiny-6\%},\cellcolor{cyan!66} 57{\tiny-3\%},\cellcolor{cyan!83} 58{\tiny-1\%},\cellcolor{cyan!100} \textbf{59*},\cellcolor{cyan!100} 59{\tiny+0\%},\cellcolor{cyan!100} 59{\tiny+0\%}
gil262,\cellcolor{cyan!64} 135{\tiny-3\%},\cellcolor{cyan!78} 137{\tiny-2\%},\cellcolor{cyan!21} 129{\tiny-7\%},\cellcolor{cyan!21} 129{\tiny-7\%},\cellcolor{cyan!92} 139{\tiny+0\%},\cellcolor{cyan!100} \textbf{140*},\cellcolor{cyan!35} 131{\tiny-6\%},\cellcolor{cyan!92} 139{\tiny+0\%}
pr299,\cellcolor{cyan!86} 149{\tiny-1\%},\cellcolor{cyan!100} \textbf{151*}{\t...,\cellcolor{cyan!73} 147{\tiny-2\%},\cellcolor{cyan!66} 146{\tiny-3\%},\cellcolor{cyan!80} 148{\tiny-1\%},\cellcolor{cyan!100} \textbf{151*},\cellcolor{cyan!100} 154{\tiny+1\%},\cellcolor{cyan!100} 152{\tiny+0\%}
lin318,\cellcolor{cyan!62} 178{\tiny-3\%},\cellcolor{cyan!89} 183{\tiny+0\%},\cellcolor{cyan!51} 176{\tiny-4\%},\cellcolor{cyan!29} 172{\tiny-6\%},\cellcolor{cyan!100} \textbf{185*}{\t...,\cellcolor{cyan!94} 184,\cellcolor{cyan!100} 191{\tiny+3\%},\cellcolor{cyan!100} 186{\tiny+1\%}
rd400,\cellcolor{cyan!85} 204{\tiny-1\%},\cellcolor{cyan!85} 204{\tiny-1\%},\cellcolor{cyan!32} 193{\tiny-6\%},\cellcolor{cyan!8} 188{\tiny-9\%},\cellcolor{cyan!100} \textbf{207*}{\t...,\cellcolor{cyan!100} \textbf{207*},\cellcolor{cyan!100} 208{\tiny+0\%},\cellcolor{cyan!100} 208{\tiny+0\%}
d493,\cellcolor{cyan!67} 295{\tiny-1\%},\cellcolor{cyan!37} 286{\tiny-4\%},\cellcolor{cyan!0} 264{\tiny-11\%},\cellcolor{cyan!0} 263{\tiny-12\%},\cellcolor{cyan!100} \textbf{305*}{\t...,\cellcolor{cyan!80} 299,\cellcolor{cyan!77} 298{\tiny+0\%},\cellcolor{cyan!57} 292{\tiny-2\%}
u574,\cellcolor{cyan!80} 303{\tiny-1\%},\cellcolor{cyan!77} 302{\tiny-2\%},\cellcolor{cyan!54} 295{\tiny-4\%},\cellcolor{cyan!57} 296{\tiny-4\%},\cellcolor{cyan!100} \textbf{309*}{\t...,\cellcolor{cyan!100} \textbf{309*},\cellcolor{cyan!100} 313{\tiny+1\%},\cellcolor{cyan!100} 312{\tiny+0\%}
u724,\cellcolor{cyan!89} 378{\tiny-1\%},\cellcolor{cyan!86} 377{\tiny-1\%},\cellcolor{cyan!55} 365{\tiny-4\%},\cellcolor{cyan!52} 364{\tiny-4\%},\cellcolor{cyan!92} 379{\tiny+0\%},\cellcolor{cyan!100} \textbf{382*},\cellcolor{cyan!100} 390{\tiny+2\%},\cellcolor{cyan!100} 393{\tiny+2\%}
pcb1173,\cellcolor{cyan!58} 555{\tiny-4\%},\cellcolor{cyan!68} 561{\tiny-3\%},\cellcolor{cyan!65} 559{\tiny-3\%},\cellcolor{cyan!63} 558{\tiny-3\%},\cellcolor{cyan!87} 572{\tiny-1\%},\cellcolor{cyan!100} \textbf{579*},\cellcolor{cyan!100} 581{\tiny+0\%},\cellcolor{cyan!100} 583{\tiny+0\%}
fl1400,\cellcolor{cyan!82} 966{\tiny-1\%},\cellcolor{cyan!72} 956{\tiny-2\%},\cellcolor{cyan!0} 791{\tiny-19\%},\cellcolor{cyan!0} 841{\tiny-14\%},\cellcolor{cyan!95} 979{\tiny+0\%},\cellcolor{cyan!100} \textbf{983*},\cellcolor{cyan!100} 1018{\tiny+3\%},\cellcolor{cyan!100} 1020{\tiny+3\%}


In [274]:
val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ")  {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ")  {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|  }
                \hline
                \multicolumn{9}{|c|}{clustering flat} \\
                \hline
                    instance & rkmn & rkmd & nckmn & nckmd & fbckmn & fbckmd & kmn & kmd \\
                \hline
                    eil101 & \cellcolor{cyan!66} 57{\tiny-3\%} & \cellcolor{cyan!66} 57{\tiny-3\%} & \cellcolor{cyan!32} 55{\tiny-6\%} & \cellcolor{cyan!66} 57{\tiny-3\%} & \cellcolor{cyan!83} 58{\tiny-1\%} & \cellcolor{cyan!100} \textbf{59*} & \cellcolor{cyan!100} 59{\tiny+0\%} & \cellcolor{cyan!100} 59{\tiny+0\%} \\ 
gil262 & \cellcolor{cyan!64} 135{\tiny-3\%} & \cellcolor{cyan!78} 137{\tiny-2\%} & \cellcolor{cyan!21} 129{\tiny-7\%} & \cellcolor{cyan!21} 129{\tiny-7\%} & \cellcolor{cyan!92} 139{\tiny+0\%} & \cellcolor{cyan!100} \textbf{140*} & \cellcolor{cyan!35} 131{\tiny-6\%} & \c